In [1]:
!pip install requests wikipedia


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import requests
import wikipedia

ENDPOINT = "http://localhost:1234/v1/chat/completions"
MODEL_ID  = "qwen/qwen3-1.7b"

WIKI_TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "search_wikipedia",
        "description": (
            "Retrieve a factual summary from Wikipedia about any person, "
            "place, technology, event, or concept."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The article title or search phrase to look up on Wikipedia.",
                }
            },
            "required": ["query"],
        },
    },
}

In [3]:
class WikipediaAgent:

    def __init__(self, endpoint: str = ENDPOINT, model: str = MODEL_ID):
        self.endpoint = endpoint
        self.model = model
        self.headers = {"Content-Type": "application/json"}
        wikipedia.set_user_agent("WikiAgent/2.0 (student_project@example.com)")

    def _fetch_wiki_summary(self, query: str) -> str:
        try:
            return wikipedia.summary(query, sentences=3)
        except wikipedia.exceptions.DisambiguationError as exc:
            suggestions = ", ".join(exc.options[:5])
            return f"'{query}' is ambiguous. Possible matches: {suggestions}"
        except wikipedia.exceptions.PageError:
            return f"No Wikipedia article found for '{query}'."
        except Exception as exc:
            return f"Wikipedia lookup failed: {exc}"

    def _call_llm(self, messages: list, use_tools: bool = False) -> dict:
        body = {
            "model": self.model,
            "messages": messages,
            "temperature": 0.7,
        }
        if use_tools:
            body["tools"] = [WIKI_TOOL_SCHEMA]
            body["tool_choice"] = "auto"

        resp = requests.post(self.endpoint, headers=self.headers, json=body)

        if resp.status_code != 200:
            raise RuntimeError(f"LM Studio returned {resp.status_code}: {resp.text}")

        return resp.json()["choices"][0]["message"]

    def _handle_tool_calls(self, messages: list, tool_calls: list) -> str:
        for tc in tool_calls:
            fn_name = tc["function"]["name"]
            fn_args = json.loads(tc["function"]["arguments"])

            if fn_name == "search_wikipedia":
                query = fn_args.get("query", "")
                print(f"  🔍 Tool call  → search_wikipedia('{query}')")

                result = self._fetch_wiki_summary(query)
                print(f"  📄 Tool result → {result}\n")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc["id"],
                    "name": fn_name,
                    "content": result,
                })

        print("Synthesizing answer…")
        final_msg = self._call_llm(messages, use_tools=False)
        return final_msg.get("content", "")

    def ask(self, question: str):
        messages = [{"role": "user", "content": question}]
        print(f"\nYou: {question}")
        print("Thinking…\n")
        try:
            reply = self._call_llm(messages, use_tools=True)
            tool_calls = reply.get("tool_calls")
            if tool_calls:
                messages.append(reply)
                answer = self._handle_tool_calls(messages, tool_calls)
            else:
                answer = reply.get("content", "")
            print(f"\nAI: {answer}\n")
        except requests.exceptions.ConnectionError:
            print("❌ Cannot reach LM Studio. Make sure the server is running at http://localhost:1234")
        except RuntimeError as err:
            print(f"❌ Error: {err}")

In [4]:
agent = WikipediaAgent()
agent.ask("Who is Albert Einstein?")


You: Who is Albert Einstein?
Thinking…

  🔍 Tool call  → search_wikipedia('Albert Einstein')
  📄 Tool result → Albert Einstein (14 March 1879 – 18 April 1955) was a German-born theoretical physicist best known for developing the theory of relativity. Einstein also made important contributions to quantum theory. His mass–energy equivalence formula E = mc2, which arises from special relativity, has been called "the world's most famous equation".

Synthesizing answer…

AI: 

Albert Einstein (14 March 1879 – 18 April 1955) was a German-born theoretical physicist renowned for his contributions to modern physics. He is best known for developing the theory of relativity, which includes **special relativity** (1905) and **general relativity** (1915). His work on the photoelectric effect earned him the Nobel Prize in Physics in 1921.

Einstein's most famous equation, **E = mc²**, expresses the relationship between mass and energy, showing that mass can be converted into energy. His ideas revol